# 1 - Downloading

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Telechargement Imagewoof (public, fast.ai). IW_ROOT = dossier contenant train/ et val/.
if env_config.in_colab():
    dl_dir = "/content"
else:
    dl_dir = os.path.join(env_config.project_home(), "data", "Imagewoof")
    os.makedirs(dl_dir, exist_ok=True)

IW_ROOT = os.path.join(dl_dir, "imagewoof2")
if not os.path.isdir(IW_ROOT):
    tgz = os.path.join(dl_dir, "imagewoof2.tgz")
    print("Telechargement Imagewoof...")
    os.system(f'wget -q https://s3.amazonaws.com/fast-ai-imageclas/imagewoof2.tgz -O "{tgz}"')
    os.system(f'tar -xzf "{tgz}" -C "{dl_dir}"')
    os.remove(tgz)
print("Imagewoof brut:", IW_ROOT)

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Decoupage Imagewoof multi-fidelite : images PROPRES (degradation a la volee).
import shutil
import random

out_dir = env_config.data_dir("Imagewoof")
raw_dir = IW_ROOT
print("Source:", raw_dir, "| Sortie:", out_dir)
if os.path.exists(out_dir):
    shutil.rmtree(out_dir)
os.makedirs(f"{out_dir}/test", exist_ok=True)
os.makedirs(f"{out_dir}/train/HF", exist_ok=True)
os.makedirs(f"{out_dir}/train/BF", exist_ok=True)

classes = os.listdir(f"{raw_dir}/train")
for cls in classes:
    shutil.copytree(f"{raw_dir}/val/{cls}", f"{out_dir}/test/{cls}")
    os.makedirs(f"{out_dir}/train/HF/{cls}", exist_ok=True)
    os.makedirs(f"{out_dir}/train/BF/{cls}", exist_ok=True)
    images = os.listdir(f"{raw_dir}/train/{cls}")
    random.seed(42)
    random.shuffle(images)
    split = int(len(images) * 0.10)
    for n in images[:split]:
        shutil.copy2(f"{raw_dir}/train/{cls}/{n}", f"{out_dir}/train/HF/{cls}/{n}")
    for n in images[split:]:
        shutil.copy2(f"{raw_dir}/train/{cls}/{n}", f"{out_dir}/train/BF/{cls}/{n}")
print("Termine ! Imagewoof propre dans:", out_dir)

In [ ]:
import sys, os, importlib
for _p in ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import env_config
importlib.reload(env_config)

# Sur Colab : zipper + uploader sur le Drive. Sur serveur/local : rien a faire.
import shutil
if env_config.in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    drive_path = "/content/drive/MyDrive/UTBM_PF22/datasets/Imagewoof"
    os.makedirs(drive_path, exist_ok=True)
    get_ipython().system('cd /content && zip -0 -r -q imagewoof_multifidelity.zip processed_multifidelity/')
    shutil.copy2("/content/imagewoof_multifidelity.zip", f"{drive_path}/dataset_multifidelity.zip")
    print("Zip Imagewoof uploade sur le Drive.")
else:
    print("Serveur/local : pas de zip necessaire (donnees deja sur le disque).")